# Pima Indians Diabetes Dataset — EDA & Feature Engineering
**Capstone Project | Healthcare Domain**  
**Task:** Binary Classification — Predict Disease Likelihood  
**Target Variable:** `Outcome` (1 = Diabetic, 0 = Not Diabetic)

---
## Step 1: Problem Understanding & Framing

In [1]:
# =============================================================================
# STEP 1: PROBLEM UNDERSTANDING & FRAMING
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════╗
║           CAPSTONE PROJECT — PROBLEM FRAMING                     ║
╚══════════════════════════════════════════════════════════════════╝

BUSINESS PROBLEM
─────────────────
Diabetes affects 830 million adults worldwide. Over 50% of cases go
undiagnosed until serious complications arise — such as kidney failure,
blindness, and amputation — costing healthcare systems significantly
more to treat than early-stage intervention.

Healthcare providers need a reliable, scalable way to identify
high-risk patients BEFORE symptoms escalate.

OBJECTIVE
─────────
Build a machine learning model to predict diabetes likelihood in
patients using routine clinical measurements, enabling early
preventive care and intervention.

DATASET
───────
Source     : Pima Indians Diabetes Database
             UCI Machine Learning Repository / Kaggle
URL        : kaggle.com/datasets/uciml/pima-indians-diabetes-database
Population : Female patients of Pima Indian heritage, aged 21+
Size       : 768 patients, 8 clinical features

TASK TYPE
─────────
  Binary Classification
  Target variable : Outcome  (1 = Diabetic, 0 = Not Diabetic)
  Input features  : 8 routine clinical measurements (no specialist tests)

SUCCESS METRICS
───────────────
  Technical (ML):
    Primary   → AUC-ROC   : measures discrimination ability (robust to imbalance)
    Secondary → Recall     : minimises missed diabetes diagnoses (False Negatives)
                F1-Score   : balances precision and recall
                Precision  : minimises unnecessary false alarms

  Business KPIs:
    → Increase early diabetes detection rate by ≥25% vs manual screening
    → Reduce late-stage complication treatment costs by targeting high-risk patients
    → Flag high-risk patients for preventive consultation before complications develop
    → Model must explain its reasoning to satisfy clinical and regulatory trust

WHY RECALL IS THE CRITICAL CLINICAL METRIC
───────────────────────────────────────────
  A missed diabetes diagnosis (False Negative) is far more dangerous
  than a false alarm (False Positive).

  False Negative → patient goes undiagnosed → years of undetected disease
                   → complications: kidney failure, blindness, neuropathy
  False Positive → unnecessary follow-up test → minor inconvenience

  Therefore, RECALL is prioritised over Accuracy throughout this project.
""")



╔══════════════════════════════════════════════════════════════════╗
║           CAPSTONE PROJECT — PROBLEM FRAMING                     ║
╚══════════════════════════════════════════════════════════════════╝

BUSINESS PROBLEM
─────────────────
Diabetes affects 830 million adults worldwide. Over 50% of cases go
undiagnosed until serious complications arise — such as kidney failure,
blindness, and amputation — costing healthcare systems significantly
more to treat than early-stage intervention.

Healthcare providers need a reliable, scalable way to identify
high-risk patients BEFORE symptoms escalate.

OBJECTIVE
─────────
Build a machine learning model to predict diabetes likelihood in
patients using routine clinical measurements, enabling early
preventive care and intervention.

DATASET
───────
Source     : Pima Indians Diabetes Database
             UCI Machine Learning Repository / Kaggle
URL        : kaggle.com/datasets/uciml/pima-indians-diabetes-database
Population : Female patient

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('All libraries imported successfully.')

---
## 2. Data Loading & First Look

In [ ]:
df = pd.read_csv('diabetes.csv')

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Features: {df.shape[1] - 1} | Target: Outcome')
df.head(10)

In [ ]:
df.info()

---
## 3. Data Dictionary

| Feature | Type | Description | Units / Range |
|---|---|---|---|
| Pregnancies | int | Number of times pregnant | 0–17 |
| Glucose | int | Plasma glucose concentration (2hr oral glucose tolerance test) | mg/dL |
| BloodPressure | float | Diastolic blood pressure | mm Hg |
| SkinThickness | int | Triceps skinfold thickness | mm |
| Insulin | float | 2-hour serum insulin | µU/mL |
| BMI | float | Body mass index | kg/m² |
| DiabetesPedigreeFunction | float | Genetic risk score based on family history | 0.0–2.5 |
| Age | int | Patient age | Years (21–81) |
| **Outcome** | int | **Target variable** | **0 = No Diabetes, 1 = Diabetes** |

---
## 4. Summary Statistics

In [ ]:
df.describe().round(2)

---
## 5. Missing Value Analysis

> ⚠️ **Important:** This dataset uses `0` as a placeholder for missing values in medically impossible fields.
> A blood pressure, BMI, or glucose of `0` is biologically impossible and must be treated as missing.

In [ ]:
# Columns where 0 is biologically impossible
zero_not_allowed = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('Zero counts in columns where 0 is impossible:')
for col in zero_not_allowed:
    n_zeros = (df[col] == 0).sum()
    pct = n_zeros / len(df) * 100
    print(f'  {col:30s}: {n_zeros:3d} zeros ({pct:.1f}%)')

In [ ]:
# Replace impossible zeros with NaN
df_clean = df.copy()
df_clean[zero_not_allowed] = df_clean[zero_not_allowed].replace(0, np.nan)

# Visualise missing values
missing = df_clean.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_clean) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(missing[missing > 0].index, missing_pct[missing > 0], color=sns.color_palette('muted'))
for bar, val in zip(bars, missing_pct[missing > 0]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val}%', ha='center', fontsize=10)
ax.set_title('Missing Values After Zero Replacement', fontsize=13)
ax.set_ylabel('Missing (%)')
ax.set_ylim(0, 55)
plt.tight_layout()
plt.show()

In [ ]:
# Impute with median (robust to outliers, clinically sensible)
for col in zero_not_allowed:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f'Imputed {col} with median: {median_val:.2f}')

print(f'\nMissing values remaining: {df_clean.isnull().sum().sum()}')

In [ ]:
# =============================================================================
# DUPLICATE ROW CHECK
# =============================================================================

duplicates = df_clean.duplicated().sum()
print(f'Duplicate rows found: {duplicates}')

if duplicates > 0:
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    print(f'Duplicates removed. New shape: {df_clean.shape}')
else:
    print('✓ No duplicate rows found — dataset is clean.')
    print(f'  Shape confirmed: {df_clean.shape}')


---
## 6. Target Variable Analysis — Class Balance

In [ ]:
counts = df_clean['Outcome'].value_counts()
labels = ['No Diabetes (0)', 'Diabetes (1)']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(labels, counts.values, color=['#5B9BD5', '#ED7D31'], width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#5B9BD5', '#ED7D31'], startangle=90)
axes[1].set_title('Class Proportions')

plt.suptitle('Target Variable Distribution', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Class imbalance ratio: {counts[0]/counts[1]:.2f}:1 (majority:minority)')
print('⚠️  Imbalanced dataset — use class_weight="balanced" or SMOTE when training models.')

---
## 7. Feature Distributions

In [ ]:
features = [col for col in df_clean.columns if col != 'Outcome']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    # Overlay by class
    df_clean[df_clean['Outcome'] == 0][col].hist(
        bins=25, ax=ax, alpha=0.6, color='#5B9BD5', label='No Diabetes')
    df_clean[df_clean['Outcome'] == 1][col].hist(
        bins=25, ax=ax, alpha=0.6, color='#ED7D31', label='Diabetes')
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle('Feature Distributions by Outcome', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Outlier Detection — Box Plots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    df_clean.boxplot(column=col, by='Outcome', ax=ax,
                     boxprops=dict(color='#5B9BD5'),
                     medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Outcome (0=No Diabetes, 1=Diabetes)')

axes[-1].set_visible(False)
plt.suptitle('Box Plots by Outcome (Outlier Detection)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Correlation Analysis

In [ ]:
corr = df_clean.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

print('Top correlations with Outcome:')
print(corr['Outcome'].drop('Outcome').sort_values(ascending=False).round(3))

---
## 10. Feature Engineering

In [ ]:
df_eng = df_clean.copy()

# --- Age Groups (clinical risk tiers) ---
df_eng['AgeGroup'] = pd.cut(df_eng['Age'],
                             bins=[20, 30, 40, 50, 60, 82],
                             labels=['21-30', '31-40', '41-50', '51-60', '61+'])

# --- BMI Categories (WHO standard) ---
df_eng['BMICategory'] = pd.cut(df_eng['BMI'],
                                bins=[0, 18.5, 25, 30, 100],
                                labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

# --- Glucose Risk Tier ---
df_eng['GlucoseRisk'] = pd.cut(df_eng['Glucose'],
                                bins=[0, 99, 125, 300],
                                labels=['Normal', 'Pre-diabetic', 'Diabetic'])

# --- Insulin-to-Glucose Ratio (proxy for insulin resistance) ---
df_eng['InsulinGlucoseRatio'] = df_eng['Insulin'] / (df_eng['Glucose'] + 1)

# --- High Risk composite flag ---
df_eng['HighRisk'] = (
    (df_eng['Glucose'] > 140) &
    (df_eng['BMI'] > 30) &
    (df_eng['Age'] > 40)
).astype(int)

print('New features added:')
print('  AgeGroup, BMICategory, GlucoseRisk (bins)')
print('  InsulinGlucoseRatio (domain-derived)')
print('  HighRisk (composite flag)')
df_eng[['AgeGroup','BMICategory','GlucoseRisk','InsulinGlucoseRatio','HighRisk']].head()

In [ ]:
# Encode categorical features for modelling
df_model = pd.get_dummies(df_eng, columns=['AgeGroup', 'BMICategory', 'GlucoseRisk'], drop_first=True)
print('Shape after encoding:', df_model.shape)

---
## 11. Feature Scaling

In [ ]:
numeric_features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'InsulinGlucoseRatio']

X = df_clean[features].copy()
y = df_clean['Outcome']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

print('Features scaled using StandardScaler (mean=0, std=1).')
print('Justification: required for distance-based models (SVM, KNN, Logistic Regression).')
X_scaled.describe().round(2)

---
## 12. Feature Selection

In [ ]:
# --- Filter Method: SelectKBest (ANOVA F-score) ---
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_scaled, y)

f_scores = pd.Series(selector.scores_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
f_scores.plot(kind='bar', color=sns.color_palette('muted'), ax=ax)
ax.set_title('Feature Importance — ANOVA F-Score (Filter Method)', fontsize=13)
ax.set_ylabel('F-Score')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nF-Scores:')
print(f_scores.round(2))

In [ ]:
# --- Embedded Method: Random Forest Feature Importances ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

rf_importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
rf_importances.plot(kind='bar', color=sns.color_palette('flare'), ax=ax)
ax.set_title('Feature Importance — Random Forest (Embedded Method)', fontsize=13)
ax.set_ylabel('Importance Score')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nRandom Forest Importances:')
print(rf_importances.round(4))

In [ ]:
# =============================================================================
# FEATURE SELECTION DECISION & JUSTIFICATION
# =============================================================================

print("""
FEATURE SELECTION DECISION
───────────────────────────
Two methods applied:
  Method 1 (Filter)   : ANOVA F-Score      — statistical test, model-independent
  Method 2 (Embedded) : Random Forest      — model-based, captures interactions

RESULTS SUMMARY
  TOP TIER    (both methods agree): Glucose, Insulin, BMI, Age
  MIDDLE TIER (both methods agree): DiabetesPedigreeFunction, SkinThickness, Pregnancies
  WEAKEST     (both methods agree): BloodPressure

DECISION: ALL 8 original features retained for modelling.

JUSTIFICATION:
  1. Small dataset (n=768) — removing features risks losing signal
     that tree-based models can still extract in combinations
  2. BloodPressure, while weakest individually, may contribute
     in interaction with Glucose and BMI inside decision trees
  3. XGBoost and Random Forest handle multicollinearity natively —
     they will down-weight irrelevant features automatically
  4. Engineered features (InsulinGlucoseRatio, HighRisk) ADD information
     beyond the originals rather than replacing them

ALTERNATIVE: If deploying Logistic Regression only, we would drop
BloodPressure and SkinThickness to reduce multicollinearity noise.
""")


---
## 13. Dimensionality Reduction — PCA

In [ ]:
pca = PCA()
pca.fit(X_scaled)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(explained_var)+1), explained_var * 100,
       alpha=0.7, color='#5B9BD5', label='Individual')
ax.plot(range(1, len(cumulative_var)+1), cumulative_var * 100,
        'o-', color='#ED7D31', label='Cumulative')
ax.axhline(y=85, linestyle='--', color='gray', alpha=0.5, label='85% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('PCA — Explained Variance', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

n_85 = np.argmax(cumulative_var >= 0.85) + 1
print(f'{n_85} components explain ≥85% of variance.')

In [ ]:
# PCA scatter — first 2 components coloured by Outcome
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=y, cmap='bwr', alpha=0.5, edgecolors='none', s=30)
plt.colorbar(scatter, ax=ax, label='Outcome')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — First Two Components (Blue=No Diabetes, Red=Diabetes)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# PCA DECISION & JUSTIFICATION
# =============================================================================

print("""
PCA FINDINGS & DECISION
─────────────────────────
Finding    : 6 principal components explain ≥90% of variance
             5 components explain ≥85% of variance

DECISION: Full 8-feature set retained for modelling (PCA not applied)

JUSTIFICATION:
  1. Dataset is small (768 rows) — PCA compression not needed for speed
     or memory. Dimensionality reduction provides most value at scale
     (thousands of features, millions of rows).

  2. Tree-based models (XGBoost, Random Forest) handle correlated features
     natively and do not require orthogonal (uncorrelated) inputs.
     PCA transformation would reduce interpretability without improving
     these models.

  3. Interpretability is critical in healthcare AI — clinicians need to
     understand predictions in terms of real features (Glucose, BMI),
     not abstract principal components.

  4. PCA scatter plot (above) confirms partial separation between diabetic
     and non-diabetic patients along PC1 — validating that genuine signal
     exists in the feature set, even if not perfectly linearly separable.

WHEN PCA WOULD BE APPLIED:
  → If using Logistic Regression or SVM on a high-dimensional dataset
  → If training was computationally expensive due to feature volume
  → For t-SNE / UMAP visualisation with more features (see extensions)

CONCLUSION: PCA used here for VISUALISATION and variance analysis only.
""")


---
## 14. EDA Summary & Key Findings

| Finding | Detail |
|---|---|
| Dataset size | 768 patients, 8 features + target |
| Class imbalance | 65.1% No Diabetes vs 34.9% Diabetes — **requires balancing** |
| Hidden missing values | Glucose, BloodPressure, SkinThickness, Insulin, BMI had impossible zeros |
| Imputation strategy | Replaced with column **median** (robust to outliers) |
| Strongest predictors | **Glucose**, **BMI**, **Age**, **DiabetesPedigreeFunction** |
| Outliers | Insulin and SkinThickness have high-value outliers — consider capping |
| PCA | ~5 components explain ≥85% of variance |
| New features | AgeGroup, BMICategory, GlucoseRisk, InsulinGlucoseRatio, HighRisk |

### Next Steps → Step 4: Model Implementation
- Baseline: Logistic Regression
- Tree-based: Decision Tree, Random Forest
- Boosting: XGBoost
- Handle imbalance: `class_weight='balanced'` or SMOTE
- Evaluate on: AUC-ROC, F1-Score, Precision-Recall Curve